# Check the published artifacts (CPU, no login)

This notebook does **not** run the model. It checks that the published files are the ones the write-up describes:

1. downloads the LoRA adapter at its pinned Hugging Face commit (`cc880b1`) and checks its size and base model,
2. downloads the 42 training rows from the dataset at its pinned commit (`ca83fce`),
3. runs this repo's GlotLID scorer (`scorers.py`, GlotLID pinned to v3) on the 42 training answers.

The training answers were written in Hinglish for this project, so 42/42 here is a check that the scorer and the pins work, not a result.
Generating replies with the adapter needs an NVIDIA GPU and access to the gated base model: see `README.md`, section 2.

Runs on CPU. First run downloads about 1.8 GB (GlotLID is 1.7 GB) and takes several minutes; later runs take seconds.

In [1]:
# Run from the repo root (local Jupyter), or anywhere on Colab: if scorers.py is not found, clone the repo.
import os, subprocess, sys
from pathlib import Path
if not Path("scorers.py").exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/kazani-351/tiny-aya-hinglish-safety"], check=True)
    os.chdir("tiny-aya-hinglish-safety")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-cpu.txt"], check=True)
sys.path.insert(0, os.getcwd())

In [2]:
import csv, json, time
from huggingface_hub import hf_hub_download
from safetensors.numpy import load_file
import scorers
ADAPTER = ("kazani351/tiny-aya-global-hinglish-lora", "cc880b1725f5beec0f4a8657f51b52eb3e10bd5a")
DATA = ("kazani351/tiny-aya-hinglish-safety-data", "ca83fcea77aa4a70e674c12425f56b05d28c64cb")

## 1. Adapter at the pinned commit

In [3]:
t = time.perf_counter()
cfg = json.load(open(hf_hub_download(ADAPTER[0], "adapter_config.json", revision=ADAPTER[1])))
w = load_file(hf_hub_download(ADAPTER[0], "adapter_model.safetensors", revision=ADAPTER[1]))
n = sum(v.size for v in w.values())
print(f"base={cfg['base_model_name_or_path']}  r={cfg['r']}  alpha={cfg['lora_alpha']}  tensors={len(w)}  params={n:,}")
assert len(w) == 504 and n == 30_228_480, "adapter differs from the published one"
del w
print(f"ok ({time.perf_counter()-t:.0f}s)")

base=CohereLabs/tiny-aya-global  r=16  alpha=32  tensors=504  params=30,228,480
ok (3s)


## 2. The 42 training rows at the pinned commit

In [4]:
path = hf_hub_download(DATA[0], "train_hinglish.csv", repo_type="dataset", revision=DATA[1])
rows = list(csv.DictReader(open(path, encoding="utf-8")))
assert len(rows) == 42
print({k: sum(r["type"] == k for r in rows) for k in sorted({r["type"] for r in rows})})

{'help_benign': 12, 'help_sounds_harmful': 12, 'refuse': 18}


## 3. GlotLID scorer on the 42 training answers

In [5]:
t = time.perf_counter()
predict = scorers.glotlid_predict()
print(f"GlotLID loaded ({time.perf_counter()-t:.0f}s)")
res = [(r["id"], r["type"], scorers.top_label(predict, r["answer"]),
        scorers.lang_kept(predict, "hinglish", r["answer"]), scorers.script_kept(predict, "hinglish", r["answer"])) for r in rows]
print(f"lang_kept {sum(x[3] for x in res):.0f}/42   script_kept {sum(x[4] for x in res):.0f}/42")
for i, typ, lab, lk, sk in res:
    if not (lk and sk):
        print("  not kept:", i, typ, "_".join(lab))

GlotLID loaded (7s)


lang_kept 42/42   script_kept 42/42
